# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Lane:** Ranking Signal Analysis (content refresh prioritization)

**Task type:** Scoring

I'm framing this as scoring rather than plain classification. The editor's real need isn't a
yes/no flag — it's an ordered list: "which of these ~30,000 items do I look at first?" A binary
label (is_declining_label) tells you whether an item is declining, but not how urgently
relative to the 16,000+ other declining items in the set. A continuous decline-risk score,
built from signals like ctr, avg_position, and scroll rate, is what actually lets an editor
sort a backlog and work top-down. The binary label I already have is the training target
underneath that score, not the deliverable itself — the deliverable is the sort order it produces.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



**What I'd predict:** A decline-risk score per item, trained against `is_declining_label`.

**Source of the label:** Defined rule, not a directly observed outcome. `is_declining_label`
is derived as `(trend_direction == 'down').astype(int)`, per the data dictionary's own
trend_direction field — a rule built on top of an underlying observed metric (trend_pct /
trend_direction), not something an editor manually tagged.

**Why it's a proxy, not ground truth:** "Declining" means the item's trend crossed into the
'down' bucket over this 90-day window. It doesn't capture *why* (seasonality, a client site
change, a real SEO problem) and it doesn't capture magnitude — a barely-declining item and a
collapsing one get the same label.

**Leakage guard:** `trend_direction` and `trend_pct` are excluded from the feature set used to
predict the score, since the label is derived directly from them — including them would let
the model "predict" the label by reading it off a reworded copy of itself.

## 3. Success metric

*One metric you can defend. What number means 'good'?*



**Metric:** Precision@K (e.g. precision@100) on the ranked priority list, using
`is_declining_label` as the evaluation ground truth.

**Why this metric:** An editor with limited time only works through the top of the list — not
all 30,000 items. Precision@K asks: of the top K items my score surfaces, how many are
actually declining? That matches the real action (review the top of a prioritized list) in a
way accuracy wouldn't — accuracy would reward the model for correctly ignoring 15,000 boring
non-declining items, which isn't the job here.

**What 'good' looks like:** Precision@100 meaningfully above the base rate (54.21% declining
overall) — say 75%+ — would mean the score is surfacing genuinely at-risk items rather than
reproducing a coin flip over the dataset's natural class balance.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



**One row = one content item**, with trailing 90-day metrics, for a given client.

In [1]:
import pandas as pd

!rm -rf flyrank-ml-internship
!git clone https://github.com/RohanKumar0095/flyrank-ml-internship.git

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df.columns = df.columns.str.strip()

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Shape (rows, columns):", df.shape)
print("Unique clients:", df['client_id'].nunique())
print("Max rows per content_id (should be 1):", df.groupby('content_id').size().max())

df.head()

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 109 (delta 28), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 1.84 MiB | 9.71 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Shape (rows, columns): (30000, 45)
Unique clients: 32
Max rows per content_id (should be 1): 1


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [2]:
print("Share of items labeled declining:", round(df['is_declining_label'].mean() * 100, 2), "%")
df['is_declining_label'].value_counts()

Share of items labeled declining: 54.21 %


,count
is_declining_label,
1,16262
0,13738


In [3]:
signal_check = df.groupby('is_declining_label')['ctr'].median()
print(signal_check)

is_declining_label
0    0.04
1    0.08
Name: ctr, dtype: float64


These numbers confirm the starter dataset has enough scale (30,000 rows, 32 clients) and a
meaningful base rate of declining items (54.2%) to make signal analysis worthwhile. Notably,
median CTR is higher for declining items (0.08%) than non-declining items (0.04%) — the
opposite of a naive assumption. This is itself the reason a fixed rule falls short: a simple
"low CTR = declining" threshold would be backwards on this data.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A hand-written rule (e.g. "flag items where CTR < X and avg_position > Y") assumes each signal
moves independently and in the expected direction. The data already breaks that assumption:
declining items have higher median CTR than non-declining ones, so a naive CTR threshold would
misclassify in the wrong direction. Real decline shows up as a combination of signals moving
together — some up, some down — not one variable crossing one line. With 45 columns across
30,000 items and 32 clients, there are too many plausible signal combinations for a human to
hand-tune thresholds for, and those relationships likely differ by client. A scored model can
weigh signals jointly and adapt as new trailing-90-day data comes in, where a fixed rule would
need constant manual re-tuning and would still miss counterintuitive cases like the CTR
pattern above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.